# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

Construim app-ul incremental, exact în ordinea în care e scris `app.py`:
funcția simplă -> mai multe input-uri -> tab Chat -> starea partajată ->
regula subiect/știre -> tab Agent -> punem tab-urile împreună -> recapitulare.

Inspirat din [Gradio Quickstart](https://www.gradio.app/guides/quickstart).
Regula tutorialului: **cât mai simplu, doar esențialul.**

> `app/app.py` este doar un strat subțire de Gradio peste `core/` (agent, graph),
> construit în cursurile C2–C7. Aici nu rescriem `core/` — îl chemăm.
> Ca să ruleze fără chei API, folosim un backend fals.

In [1]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\vitok\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# o singură dată:  %pip install -q gradio
import gradio as gr
print("Gradio", gr.__version__)

C:\Users\vitok\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.13.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [3]:
def saluta(nume):
    return "Salut, " + nume

gr.Interface(fn=saluta, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [4]:
def combina(text, optiune, numar):
    return f"[{optiune} @ {numar}] {text}"

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label="Text"),
            gr.Dropdown(["a", "b"], value="a", label="Opțiune"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Număr")],
    outputs=gr.Textbox(label="Rezultat"),
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

In [9]:
## 3. Backend fals (ca să rulăm fără chei API)
from pathlib import Path
import os

from IPython.display import Image, display

# Setăm manual rădăcina proiectului.
PROJECT_ROOT = Path(r"D:\SOCASIS\Ingineria AI\echochamber-project-team-4")
os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd())
# În `app.py` real, sus, sunt 2 importuri din `core/` (construite în C6–C7):

# ```python
from core.agent import generate_agent_response   # un agent RAG
from core.graph import run_thread                 # dezbatere multi-agent


# 
#Aici le înlocuim cu funcții-jucărie. Restul codului rămâne identic ca structură.

Project root: D:\SOCASIS\Ingineria AI\echochamber-project-team-4


C:\Users\vitok\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [10]:
def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {"anti_sistem": "Instituțiile par din nou rupte de oameni.",
            "anti_suveranist": "Să discutăm pe baza procedurilor."}
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [("Anti-sistem", "anti_sistem"), ("Anti-suveranist", "anti_suveranist")]
print("backend fals pregătit")

backend fals pregătit


## 4. Primul tab real: Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`. O reproducem
cu `fake_llm`.

In [11]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Întrebare / prompt", lines=4),
    outputs=gr.Textbox(label="Răspuns", lines=10),
    title="Chat",
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și (opțional) încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

- `CFG` = provider / model / temperatură
- `ART` = textul + titlul știrii încărcate

Setări **scrie** în ele, restul tab-urilor **citesc**. (Alternativa „canonică"
ar fi `gr.State`; varianta minimă alege simplitatea.)

Truc din `app.py`: provider + model sunt **un singur dropdown**
(`"provider|model"`) — imposibil să fie nepotrivite.

In [12]:
CFG = {"provider": "gemini", "model": "gemini-2.5-flash-lite", "temp": 0.3}
ART = {"text": "", "title": ""}

MODEL_CHOICES = [("gemini · gemini-2.5-flash-lite", "gemini|gemini-2.5-flash-lite"),
                 ("deepseek · deepseek-chat",       "deepseek|deepseek-chat")]

def setup(model_choice, temperature, fake_url):
    provider, model = model_choice.split("|", 1)     # despărțim "provider|model"
    CFG.update(provider=provider, model=model, temp=temperature)
    if fake_url.strip():
        ART.update(text=f"Text fals al știrii de la {fake_url}", title=fake_url)
        return f"Setări salvate. Știre ACTIVĂ: {fake_url}"
    ART.update(text="", title="")
    return f"Setări salvate ({provider} · {model}). Fără știre."

gr.Interface(
    fn=setup,
    inputs=[gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1],
                        label="Provider · Model"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
            gr.Textbox(label="URL știre (gol = fără știre)")],
    outputs=gr.Textbox(label="Stare"),
    title="Setări",
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## 6. Regula cheie: subiectul intră *peste* știre

`_subject()` decide ce primește un agent:

- **știre + subiect** -> vorbim despre subiect, dar **în contextul știrii**
- **doar știre** -> vorbim despre știre
- **doar subiect** -> vorbim doar despre subiect

In [13]:
def _subject(typed):
    typed = (typed or "").strip()
    news = ART["text"].strip()
    if news and typed:
        return f"{typed}\n\n[În contextul acestei știri:]\n{news[:600]}"
    if news:
        return news[:700]
    return typed

ART.update(text="Știre despre UE și energie.")
print(_subject("Bolojan"))     # subiect peste știre
ART.update(text="")
print(_subject("Bolojan"))     # doar subiect

Bolojan

[În contextul acestei știri:]
Știre despre UE și energie.
Bolojan


In [14]:
ART

{'text': '',
 'title': 'https://recorder.ro/stirile-zilei/18-mai-2026-cotroceni-consultari-in-gol/'}

## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului. În `app.py` real,
`fake_agent` e `generate_agent_response` din `core.agent` (C6).

In [15]:
def agent(text, slug):
    s = _subject(text)
    if not s.strip():
        return "Încarcă o știre sau scrie un subiect."
    return fake_agent(slug, s)               # în app: generate_agent_response(...)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label="Subiect (intră peste știre, dacă e încărcată)",
                       lines=3),
            gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    outputs=gr.Textbox(label="Comentariu", lines=10),
    title="Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** au exact același tipar:
o funcție + un `gr.Interface`. Doar funcția diferă (rezumat / loop pe roluri /
`core.graph.run_thread`).

## 8. Punem tab-urile împreună

`app.py` are 6 tab-uri cu o **temă comună**. `gr.TabbedInterface` nu acceptă
`theme=` pe toate versiunile, așa că facem ce face el intern: un `gr.Blocks`
cu temă, `gr.Tabs`, și randăm fiecare `Interface` cu `.render()`.

In [16]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="Setări")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

TABS = [("Setări", tab_setup), ("Chat", tab_chat), ("Agent", tab_agent)]

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown("# EchoChamber Studio")
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

C:\Users\vitok\AppData\Local\Temp\ipykernel_16580\3885253093.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul exact din `app.py`. În aplicația reală sunt 6 tab-uri în
loc de 3, iar funcțiile cheamă `core/` în loc de `fake_*`.

# TODO - Tema 3
## TODO final — modifică aplicația Gradio din notebook
În ultima parte a notebook-ului, continuă de la aplicația Gradio construită mai sus și adaugă o mică extensie individuală.
Nu trebuie să refaci aplicația de la zero. Modifică direct codul existent din notebook.
Trebuie să adaugi cel puțin **3 modificări vizibile**:
1. **Un tab nou**
   Exemplu: `Despre`, `Etică`, `Ajutor`, `Rezumat`, `Export`, `Analiză`.
   Tabul trebuie să apară în aplicație când rulezi ultima celulă.
2. **O funcție simplă nouă**
   Exemplu:
   - rezumă textul introdus;
   - numără cuvintele;
   - curăță textul;
   - transformă răspunsul într-o variantă mai scurtă;
   - formatează rezultatul pentru copiere.
3. **Un element de design**
   Exemplu:
   - titlu mai bun;
   - subtitlu;
   - emoji pentru taburi;
   - altă temă Gradio;
   - card vizual pentru rezultat;
   - layout mai clar cu `gr.Row()` sau `gr.Column()`.
Opțional, poți adăuga și:
4. **O opțiune de utilizator**
   Exemplu:
   - dropdown pentru tipul de răspuns;
   - slider pentru lungimea răspunsului;
   - selector pentru ton;
   - checkbox pentru răspuns scurt/lung.
   
### Cerință minimă
La final, aplicația trebuie să ruleze în notebook și modificările trebuie să fie vizibile în interfață.
### Scrie sub cod, în 3–4 propoziții:
- Ce am adăugat: 

Am adaugat doua taburi noi in interfata ("Utilitar Text" si "Rezumat"), un checkbox pentru formatarea textului si un buton dinamic sub forma de emoji pentru comutarea rapida a temei vizuale.

- Ce funcție nouă am creat:

Am creat functiile utilitar_text (pentru numarat cuvinte) si rezuma_text (pentru scurtarea textelor lungi).

- Ce element de design am modificat:

Am restructurat antetul cu gr.Row() si gr.Column(), am pus emoji-uri pe taburi si am introdus CSS personalizat pentru a face butonul temei complet transparent, lasand vizibil doar emoji-ul (☀️ / 🌙).
- Ce aș îmbunătăți dacă aș continua aplicația:

In [24]:
import gradio as gr

# 1. FUNCTIE NOUA: Utilitar pentru a procesa texte (numara cuvinte si formateaza)
def utilitar_text(text, majuscule):
    if not text.strip():
        return "Introduceti un text pentru a fi procesat."
    numar_cuvinte = len(text.split())
    rezultat = f"Statistici: Textul contine {numar_cuvinte} cuvinte.\n\n"
    if majuscule:
        rezultat += f"Text formatat:\n{text.upper()}"
    else:
        rezultat += f"Text original curatat:\n{' '.join(text.split())}"
    return rezultat

# 2. FUNCTIE NOUA: Functie simpla pentru rezumat text
def rezuma_text(text):
    if not text.strip():
        return "Introduceti un text pentru a fi rezumat."
    
    cuvinte = text.split()
    if len(cuvinte) <= 10:
        return f"Textul este deja foarte scurt:\n\"{text}\""
    
    scurtat = " ".join(cuvinte[:12]) + "..."
    return f"📌 Rezumat automat (Ideea principala):\n\"{scurtat}\"\n\n[Nota: Pentru rezumate avansate, conectati aceasta functie la LLM]"

# Script JavaScript care schimba tema si returneaza DOAR emoji-ul
toggle_theme_js = """
(btn_text) => {
    const body = document.querySelector('body');
    body.classList.toggle('dark');
    
    if (body.classList.contains('dark')) {
        return "☀️";
    } else {
        return "🌙";
    }
}
"""

# 3. CSS PERSONALIZAT: Face butonul transparent si ii da un efect la hover
css_personalizat = """
#buton_tema {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    font-size: 35px !important; /* Face emoji-ul mai mare */
    min-width: auto !important;
    padding: 0 !important;
}
#buton_tema:hover {
    transform: scale(1.1); /* Efect de marire la trecerea cu mouse-ul */
    background: transparent !important;
}
"""

# Interfetele tale originale (presupunand ca setup, chat, agent sunt definite mai sus in notebook)
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatura"),
     gr.Textbox(label="URL stire")],
    gr.Textbox(label="Stare"), title="Setari")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Raspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

# Interfata pentru primul TAB utilitar
tab_utilitar = gr.Interface(
    fn=utilitar_text,
    inputs=[gr.Textbox(label="Text pentru analiza", lines=3),
            gr.Checkbox(label="Transforma tot textul in MAJUSCULE", value=False)],
    outputs=gr.Textbox(label="Rezultat", lines=5),
    title="Utilitar Text"
)

# Interfata pentru TAB-ul de Rezumat
tab_rezumat = gr.Interface(
    fn=rezuma_text,
    inputs=gr.Textbox(label="Introduceti textul lung aici", lines=6, placeholder="Scrie sau lipeste un text lung..."),
    outputs=gr.Textbox(label="Text Rezumat", lines=4),
    title="Rezumat Rapid"
)

TABS = [
    ("⚙️ Setari", tab_setup), 
    ("💬 Chat", tab_chat), 
    ("🤖 Agent", tab_agent), 
    ("🛠️ Utilitar Text", tab_utilitar),
    ("📝 Rezumat", tab_rezumat)
]

# DESIGN: Am adaugat parametrul css=css_personalizat
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"), css=css_personalizat) as demo:
    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("# 🎙️ EchoChamber Studio PRO")
            gr.Markdown("### Interfata avansata pentru simularea si analiza discursului politic (Versiune extinsa)")
        with gr.Column(scale=1, min_width=50):
            # DESIGN: Butonul are acum un elem_id pentru a fi controlat din CSS si contine doar emoji-ul
            theme_btn = gr.Button("🌙", elem_id="buton_tema")
            
    theme_btn.click(None, inputs=[theme_btn], outputs=[theme_btn], js=toggle_theme_js)
    
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

C:\Users\vitok\AppData\Local\Temp\ipykernel_16580\553532508.py:98: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"), css=css_personalizat) as demo:


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


## 9. Recapitulare

**Ce face Gradio (tot tutorialul, pe scurt):**

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `CFG` / `ART` (dict-uri de modul) = starea partajată: Setări scrie, restul citesc.
4. `_subject()` = regula subiect-peste-știre.
5. `gr.Blocks` + `gr.Tabs` + `.render()` = cele 6 tab-uri cu temă comună.

**Dependențe (din structura repo):** `app/app.py` cheamă doar `core/` —
nu rescrie nimic.

| Tab(uri) | Funcție în app.py | Backend | Curs |
|---|---|---|---|
| Setări · Chat · Rezumat | `setup` · `chat` · `summary` | apel LLM direct | C2 |
| Agent | `agent` -> `_agent` | `core.agent` (FAISS + rol) | C5 + C6 |
| Toți agenții | `all_agents` | loop pe `roles.yaml` -> `core.agent` | C6 |
| Dezbatere | `debate` | `core.graph.run_thread` (LangGraph) | C7 |

`core.agent` -> `core.retriever` (FAISS) + `roles.yaml` + LLM.
`core.graph` orchestrează `core.agent` (round-robin). Singura punte offline->runtime:
vectorstore-urile construite offline, citite de retriever la fiecare cerere.

**Mesajul cheie:** aplicația nu e un proiect nou. E un strat subțire Gradio
peste funcțiile din C2–C7. Fiecare tab = un buton peste o funcție de curs.